# N00: Metadata + Sentence-BERT Baseline (Experiment 0)

Demonstration of the **core bottleneck** in metadata-only search: metadata describes broad categories (*genre*, *style*, *artist*),
but completely lacks visual content descriptions (*turbulent sea*, *snowy mountain*, *fruit basket*).

### Split Hygiene (ADR 0003 & ADR 0005)
- **Corpus:** Museum A validation split (480 paintings).
- **Museum B (test):** strictly locked for the final presentation.
- **Results:** automatically tracked in `results/nlp_experiments.csv`.


In [ ]:
import pandas as pd
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from deep_learning_project_waerniers_lamy.nlp.utils import load_paintings, evaluate_retrieval_benchmark, save_nlp_result, plot_retrieved_paintings
from deep_learning_project_waerniers_lamy.nlp.benchmark import get_benchmark_queries

# 1. Load Museum A validation split
paintings = load_paintings(split="val")
print(f"Loaded {len(paintings)} paintings from Museum A validation split.")
paintings[["id", "genre", "style", "artist"]].head()


## 2. Format searchable metadata strings and compute embeddings

We concatenate the available metadata: `"{genre} by {artist}, {style} style"`.

In [ ]:
def format_metadata(row):
    artist = str(row["artist"]).replace("-", " ")
    style = str(row["style"]).replace("_", " ")
    genre = str(row["genre"]).replace("_", " ")
    if "unknown" in artist.lower():
        return f"{genre}, {style} style"
    return f"{genre} by {artist}, {style} style"

texts = [format_metadata(r) for _, r in paintings.iterrows()]
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
print(f"Metadata embeddings: {tuple(doc_embeddings.shape)}")


## 3. Benchmark evaluation on Museum A

We evaluate on the fixed benchmark query deck covering:
- **Visual concepts** (absent from metadata)
- **Compound queries**
- **Explicit metadata queries**

In [ ]:
painting_ids = paintings["id"].tolist()

def retrieve(query: str, top_k: int = 10):
    query_emb = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = torch.matmul(doc_embeddings, query_emb).cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [painting_ids[i] for i in top_indices]

benchmark = get_benchmark_queries()
eval_benchmark = [q for q in benchmark if q.get("relevant_ids")]
categories = sorted(set(q["category"] for q in eval_benchmark))

print("-" * 65)
for cat in categories:
    cat_queries = [q for q in eval_benchmark if q["category"] == cat]
    res = evaluate_retrieval_benchmark(retrieve, cat_queries, k_values=(5, 10))
    print(f"{cat:<18} | Recall@5: {res['recall_at_5']*100:5.1f}% | Recall@10: {res['recall_at_10']*100:5.1f}% | MRR: {res['mrr']:0.3f}")
print("-" * 65)


## 4. Visual example: why metadata fails on visual queries

When searching for *"turbulent stormy sea with large waves"*, the model matches random landscapes and cityscapes simply because "landscape" is close in embedding space, despite no sea being depicted.

In [ ]:
sample_query = "turbulent stormy sea with large waves"
top_ids = retrieve(sample_query, top_k=5)
retrieved_df = paintings[paintings["id"].isin(top_ids)].set_index("id").loc[top_ids].reset_index()
plot_retrieved_paintings(sample_query, retrieved_df, top_k=5)
